### Learning Instruction Finetuning

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{project_root}/execution_results/runs"

In [2]:
from pathlib import Path
import os
import socket
import sys
from dotenv import load_dotenv
from pathlib import Path
import sys

In [3]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
load_dotenv((project_root /'env.env').resolve())
data_dir = project_root / "datasets"

Project Root: /storage/ice1/1/0/vchopra37/projects/learning_env


In [4]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "pace" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [5]:
import torch
from datasets import Dataset as hf_data
from torch.utils.data import Dataset
import tiktoken
import numpy, tqdm
import matplotlib.pyplot as plt
%matplotlib inline

In [6]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [7]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [8]:

from model_training.framework.data.post_train.instruction_tuning.raw_data.simple_example.instruct_data import download_and_load_file
from model_training.framework.data.post_train.instruction_tuning.prompt_style.styles import SFTDatasetPrepare, AlpacaFormatOld
from model_training.framework.data.post_train.dataloader import InstructionDataset, custom_collate_fn, create_instruction_dataloader

    ## Fix this import
    
    # raise Exception("Not Running on Pace")
        # from src.model import GPT

In [9]:
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, f"{props.total_memory / 1024**3:.1f} GiB")

GPU count: 1
0 NVIDIA A100-PCIE-40GB 39.5 GiB


### Step-1:-  Preparing the dataset

We are using Alpaca Format, every models have their own format, so the format of the tuning can differ. <br>
THis format is called ALPACA format{instruction, input output}... <br>
 There are different formats that exist and the data format doesn't matter, we need to take prompt style templates for instruction tuning. We need to apply those prompt templates for instruction tuning. 

In [10]:
file_name = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(data_dir=(data_dir / "instruction_tuning"), file_name=file_name, url = url)
print("Number of entries:", len(data))
data[50] , data[999]

Number of entries: 1100


({'instruction': 'Identify the correct spelling of the following word.',
  'input': 'Ocassion',
  'output': "The correct spelling is 'Occasion.'"},
 {'instruction': "What is an antonym of 'complicated'?",
  'input': '',
  'output': "An antonym of 'complicated' is 'simple'."})

In [11]:
# alpaca = AlpacaFormatOld(data)

In [12]:
# ## Example of format
# print(alpaca.format_input(instruction=data[50]['instruction'], in_text=data[50]['input']))
# print("=============================================")
# print(alpaca.detailed_response(out_text = data[50]['output']))

In [13]:
# sample = (alpaca.complete_response(instruction=data[999]['instruction'], in_text=data[999]['input'], out_text=data[999]['output']))
# print(sample)

In [14]:
dataset  = hf_data.from_list(data)
sft_data= SFTDatasetPrepare(model_name=MODEL_NAME, template="model",)
prepared = sft_data.prepare(dataset)

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

In [15]:
print(prepared[0])
print(sft_data.preview(prepared[0]))

{'prompt': [{'content': 'Evaluate the following phrase by transforming it into the spelling given.\n\nInput:\nfreind --> friend', 'role': 'user'}], 'completion': [{'content': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".', 'role': 'assistant'}]}
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Evaluate the following phrase by transforming it into the spelling given.

Input:
freind --> friend<|im_end|>
<|im_start|>assistant
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".<|im_end|>



#### Spliting the data 

In [16]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = prepared.select(range(train_portion))
test_data = prepared.select(
    range(train_portion, train_portion + test_portion)
)
val_data = prepared.select(
    range(train_portion + test_portion, len(prepared))
)

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


### Step-2: - Prepare the Training Set Batches

#### Setting up Tokenizer

In [17]:
# tokenizer = tiktoken.get_encoding("gpt2")
# pad_token_id= tokenizer.encode('<|endoftext|>', allowed_special={"<|endoftext|>"})[0]
# pad_token_id
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

pad_token_id = tokenizer.pad_token_id
pad_token_id

151643

In [18]:
train_dataset = InstructionDataset(
    ds=train_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)


In [19]:
val_dataset = InstructionDataset(
    ds=val_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)
test_dataset = InstructionDataset(
    ds=test_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)

#### Sample Data Prepare

In [20]:

inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
custom_collate_fn(batch, pad_token_id=pad_token_id)

(tensor([[     0,      1,      2,      3,      4],
         [     5,      6, 151643, 151643, 151643],
         [     7,      8,      9, 151643, 151643]]),
 tensor([[     1,      2,      3,      4, 151643],
         [     6, 151643,   -100,   -100,   -100],
         [     8,      9, 151643,   -100,   -100]]))

In [21]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],  # 1st training example
     [-0.5, 1.5]]  # 2nd training example
)
targets_1 = torch.tensor([0, 1])


loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # New 3rd training example
)
targets_2 = torch.tensor([0, 1, 1])

loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)
targets_3 = torch.tensor([0, 1, -100])

loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
tensor(0.7936)
tensor(1.1269)
loss_1 == loss_3: tensor(True)


### Creating, Training and Testing Dataset

In [22]:
from functools import partial
# Changing the defaults at runtime
custom_collate_fn1 = partial(custom_collate_fn, device=device, allowed_max_length = 1024, pad_token_id=pad_token_id)

In [23]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = create_instruction_dataloader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)
val_loader = create_instruction_dataloader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_loader = create_instruction_dataloader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [24]:
print("Train loader:")
for inputs, targets in train_loader:
    sample_input , sample_target = inputs, targets
    print(inputs.shape, targets.shape)
    break

Train loader:
torch.Size([8, 59]) torch.Size([8, 59])


### Step-4: - Loading the Model Testing inference

In [25]:
from model_training.framework.model.load_model.gpt2.model import return_model
from model_training.framework.model.load_model.gpt2.configs import load_model_config
from model_training.framework.model.components.inference.inference import gen_sample
from model_training.framework.model.tokenizer.tokenizer_utils import text_to_token_ids, token_ids_to_text

In [26]:
config =load_model_config(dataset_max_len=1024)
# MODEL_NAME = 'gpt2'

model = return_model(MODEL_NAME)

# model = GPT.from_pretrained(config, "gpt2")
# print(f"Model Loaded: {model}")
# model.eval()
# model = model.to(device)

loading wights from pretrained gpt: gpt2-small
{'n_layer': 12, 'n_head': 12, 'emb_dim': 768}


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [27]:
# import inspect
# inspect.signature(tokenizer.encode)

In [28]:
model.eval()
text_1 = "Every effort moves you"
in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
                            #   allowed_special = {"<|endoftext|>"}
                              )
in_tokens = in_tokens.to(next(model.parameters()).device)
print(in_tokens.shape)  # [1, sequence_length]

torch.Size([1, 4])


In [ ]:
out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
print(token_ids_to_text(out_tokens, tokenizer))

### SFT-Training

In [ ]:
import torch.nn.functional as F

model = model.to(device).eval()
sample_input = sample_input.to(device)
sample_target = sample_target.to(device)

with torch.no_grad():
    out = model(input_ids=sample_input)
    loss = F.cross_entropy(
        out.logits.reshape(-1, out.logits.size(-1)),
        sample_target.reshape(-1),
        ignore_index=-100,
    )

print(loss.item())

In [ ]:
from model_training.framework.train.core.train import train_model
from model_training.framework.train.core.config import TrainConfig
from model_training.framework.train.utils.optimizer import configure_optimizer
from model_training.framework.train.utils.lr_scheduler import cosine_lr

In [ ]:
seq_length = 1024
total_batch_size = batch_size * seq_length
micro_batch_size = 4
grad_accum_steps = total_batch_size // (micro_batch_size * seq_length )
# seq_length = 1024
# micro_batch_size = train_loader.batch_size  # 1
# grad_accum_steps = 8
num_epochs = 2

In [ ]:
from dataclasses import  dataclass
from typing import Any

@dataclass
class TrainComponents:
    optimizer:Any
    dataloader:Any
    scheduler:Any
    

In [ ]:


train_config = TrainConfig(
    num_steps=(num_epochs * len(train_loader.loader)) // grad_accum_steps,
    learning_rate=1e-5,
    total_batch_size=micro_batch_size * seq_length * grad_accum_steps,
    micro_batch_size=micro_batch_size,
    seq_length=seq_length,
    accumulation_steps=grad_accum_steps,
    max_lr=1e-5,
    min_lr=1e-5,
    warmup_steps=0,
    weight_decay=0.1,
    device=device,
    use_compile=False,
    precision="fp32",
)

In [ ]:
torch.manual_seed(123)

optimizer = configure_optimizer(model, train_config.weight_decay, train_config.learning_rate, master_process= True, device=train_config.device)

In [ ]:
train_comp = TrainComponents(optimizer=optimizer , dataloader=train_loader, scheduler= cosine_lr)
train_comp

In [ ]:
torch.set_float32_matmul_precision('high') ## TF16 -> Model Training 

In [ ]:
print(train_config.accumulation_steps)

In [ ]:
from datetime import datetime
from model_training.framework.train.tracking.logger import ExperimentLogger

run_name = datetime.now().strftime("%Y%m%d_%H%M%S")

logger = ExperimentLogger(
    project="sft-experiments",
    log_dir=str(project_root / "execution_results" / "runs" / run_name),
)

In [ ]:
model = train_model(model, train_components=train_comp, train_config=train_config, logger = logger)

In [ ]:
# # text_1 = "What is your name ? "
# text_1 = "Every effort moves you"
# in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
#                             #   allowed_special = {"<|endoftext|>"}
#                               )
# in_tokens=in_tokens.to(device)
# print(in_tokens.shape)  # [1, sequence_length]


# out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
# print(token_ids_to_text(out_tokens, tokenizer))

### Setting up LORA 

In [ ]:
q_proj = model.model.layers[0].self_attn.q_proj

print("Layer:", q_proj)
print("Weight shape:", tuple(q_proj.weight.shape))
print("Bias shape:", tuple(q_proj.bias.shape) if q_proj.bias is not None else None)
print("Trainable parameters:", sum(p.numel() for p in q_proj.parameters() if p.requires_grad))

In [ ]:
import torch
import torch.nn as nn

rank = 4
A = nn.Parameter(torch.randn(rank, q_proj.in_features))
B = nn.Parameter(torch.zeros(q_proj.out_features, rank))

print(A.shape, B.shape, (B @ A).shape)

In [ ]:
x = torch.randn(1, 2048)
x.shape

In [ ]:
low_rank = x @ A.T
adjustment = low_rank @B.T

In [ ]:
print("Low-rank shape:", low_rank.shape)
print("Adjustment shape:", adjustment.shape)
print("Adjustment is zero:", torch.all(adjustment == 0).item())

In [ ]:
alpha =4 
x = x.to(device = q_proj.weight.device, dtype = q_proj.weight.dtype)

In [ ]:
base_output = q_proj(x)

In [ ]:
adjustment = (x @A.to(x).T) @ B.to(x).T
lora_output = base_output  + (alpha /rank) * adjustment